# Notebook 3: Training Loop Basics
**Estimated time:** 35-45 min
**Prerequisites:** Notebooks 1-2

---
## What you will learn
1. Loss functions — what they measure and which to use when
2. Optimizers — SGD vs Adam and how they work
3. The canonical 5-step training loop
4. Validation loop and overfitting detection
5. Plotting training curves

## 1. Loss Functions

A **loss function** measures how wrong your model is. The smaller the loss, the better.

Think of it like a **score in reverse** — 0 is perfect, bigger is worse.

| Task | Loss function | PyTorch class |
|------|--------------|---------------|
| Regression (predict a number) | Mean Squared Error | `nn.MSELoss()` |
| Binary classification (yes/no) | Binary Cross Entropy | `nn.BCEWithLogitsLoss()` |
| Multi-class classification | Cross Entropy | `nn.CrossEntropyLoss()` |

**Cross-entropy explained simply:**
Imagine your model says "I'm 90% sure this is a cat." If it IS a cat → small loss.
If it's actually a dog → large loss. Cross-entropy punishes confident wrong answers hard.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# --- MSE Loss ---
pred = torch.tensor([2.5, 0.0, 2.0])
true = torch.tensor([3.0, -0.5, 2.0])
mse = nn.MSELoss()
print(f'MSE Loss: {mse(pred, true):.4f}')

# --- Cross Entropy Loss ---
# Input: raw logits (un-normalized scores) — shape (batch, num_classes)
# Target: class indices — shape (batch,)
logits = torch.tensor([[2.0, 1.0, 0.1],    # sample 1: high score for class 0
                        [0.1, 0.5, 3.0]])   # sample 2: high score for class 2
targets = torch.tensor([0, 2])              # correct classes
ce = nn.CrossEntropyLoss()
print(f'CrossEntropy Loss: {ce(logits, targets):.4f}')  # should be small (correct!)

# Wrong prediction example
targets_wrong = torch.tensor([2, 0])       # both wrong
print(f'CrossEntropy (wrong): {ce(logits, targets_wrong):.4f}')  # larger

## 2. Optimizers

An optimizer takes the gradients and decides **how** to update the weights.

**SGD (Stochastic Gradient Descent):**
Simple rule: `weight = weight - lr * gradient`
Like walking straight downhill. Works, but can be slow or get stuck.

**Adam (Adaptive Moment Estimation):**
Adjusts the learning rate per parameter based on recent gradient history.
Think of it as a smart GPS that knows which roads are bumpy — it adapts its speed.
Almost always the better default choice.

In [ ]:
import torch.optim as optim

model = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

# SGD: just need learning rate
optimizer_sgd = optim.SGD(model.parameters(), lr=0.01)

# Adam: also works well with default betas
optimizer_adam = optim.Adam(model.parameters(), lr=1e-3)

# AdamW: Adam with weight decay (regularization) — often preferred
optimizer_adamw = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

print('Optimizers created. Using AdamW for the rest of this notebook.')

## 3. The 5-Step Training Loop

Every PyTorch training loop follows this exact pattern. Memorize it.

```
for each batch:
    1. zero_grad()     — clear old gradients
    2. forward pass    — compute predictions
    3. compute loss    — measure error
    4. backward()      — compute gradients
    5. optimizer.step()— update weights
```

**Why zero_grad() first?**
Because PyTorch *accumulates* gradients (adds to existing ones). If you forget to zero them, your updates are corrupted by gradients from previous steps.

In [ ]:
# Full example: classify 2D points into 2 classes
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

torch.manual_seed(42)

# Generate data
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
X_val   = torch.tensor(X_val_np,   dtype=torch.float32)
y_val   = torch.tensor(y_val_np,   dtype=torch.long)

print(f'Train samples: {len(X_train)}')
print(f'Val   samples: {len(X_val)}')

# Visualize
plt.figure(figsize=(6, 4))
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap='bwr', alpha=0.5, edgecolors='k', s=20)
plt.title('Moon dataset')
plt.show()

In [ ]:
# Build model, loss, optimizer
model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 2)   # 2 output classes
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_losses = []
val_losses   = []
val_accs     = []

EPOCHS = 100

for epoch in range(EPOCHS):
    # ---- TRAINING MODE ----
    model.train()   # tells dropout/batchnorm to behave as in training

    # 1. Zero gradients
    optimizer.zero_grad()

    # 2. Forward pass
    logits = model(X_train)

    # 3. Compute loss
    loss = criterion(logits, y_train)

    # 4. Backward pass
    loss.backward()

    # 5. Update weights
    optimizer.step()

    train_losses.append(loss.item())

    # ---- VALIDATION ----
    model.eval()    # tells dropout/batchnorm to behave as in evaluation
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss   = criterion(val_logits, y_val)
        preds      = val_logits.argmax(dim=1)
        acc        = (preds == y_val).float().mean()

    val_losses.append(val_loss.item())
    val_accs.append(acc.item())

    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f} | Val Acc: {acc:.4f}')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train')
ax1.plot(val_losses,   label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves')
ax1.legend()

ax2.plot(val_accs)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize decision boundary
def plot_decision_boundary(model, X, y):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        preds = model(grid).argmax(dim=1).numpy()
    zz = preds.reshape(xx.shape)
    plt.contourf(xx, yy, zz, alpha=0.3, cmap='bwr')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolors='k', s=20)

model.eval()
plt.figure(figsize=(6, 5))
plot_decision_boundary(model, X_np, y_np)
plt.title('Learned Decision Boundary')
plt.show()

## 4. train() vs eval() Mode

`model.train()` and `model.eval()` are important — they affect:
- **Dropout**: active during training (randomly drops neurons), disabled during eval
- **Batch Normalization**: uses batch statistics during training, running statistics during eval

**Rule of thumb:**
- Call `model.train()` before your training loop
- Call `model.eval()` before validation/test
- Use `with torch.no_grad():` during eval to save memory (no gradient computation needed)

---
## YOUR TURN — Mini Project

**Task:** Train a classifier on sklearn's `make_circles` dataset.

**Steps:**
1. Generate data with `make_circles(n_samples=800, noise=0.1, factor=0.5)`
2. Split into train/val (80/20)
3. Build a model with at least 3 linear layers
4. Train for 150 epochs using Adam
5. Plot the training/validation loss curves
6. Plot the decision boundary
7. Print final validation accuracy

**Challenge:** Try training without `model.train()`/`model.eval()` calls and see if it makes a difference here (it won't — there's no dropout or batchnorm yet — note this for Notebook 5!)

In [ ]:
# YOUR CODE HERE
from sklearn.datasets import make_circles

torch.manual_seed(0)

# 1. Generate data

# 2. Convert to tensors and split

# 3. Build model

# 4. Loss and optimizer

# 5. Training loop (150 epochs)

# 6. Plot curves

# 7. Plot decision boundary and print accuracy